In [2]:
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
import math
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import csv
import time

#1. Getting prices
def prices(startd, endd, tic):
    info = []
    key = '&apikey=ZKMMTO1ATDBLXH2K'
    ticker = '&symbol=' + str(tic)
    endpoint = 'function=TIME_SERIES_DAILY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker + size + key

    r = requests.get(url)
    if r.status_code == 200:
        print('connection successful')
        data = r.json()
        r1 = data.get('Time Series (Daily)', {})
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info


def compute_log_returns(data, freq='daily'):
    """
    Computes log returns at specified frequency.

    Parameters:
    -----------
    data : list of lists
        Price data: [[ticker, date, adjusted_close], ...]
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'

    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return], ...]
    """
    sorted_data = sorted(data, key=lambda x: x[1])
    data_with_logret = []

    if freq == 'daily':
        for i in range(len(sorted_data)):
            if i == 0:
                data_with_logret.append(sorted_data[i] + [None])
            else:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[i-1][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])

    elif freq == 'weekly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 7:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    elif freq == 'monthly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 30:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    elif freq == 'yearly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 360:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    return data_with_logret


def download_and_export(ticker, startd, endd, freq='daily', path=None):
    """
    Downloads price data, computes log and simple returns, and exports to CSV.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g. 'NVDA')
    startd : str
        Start date 'YYYY-MM-DD'
    endd : str
        End date 'YYYY-MM-DD'
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    path : str
        File path for CSV output. Defaults to '{ticker}_returns.csv'

    Returns:
    --------
    list of lists: [[ticker, date, price, log_return, simple_return], ...]
    First row is a header row.
    """
    if path is None:
        path = f"{ticker}_returns.csv"

    raw = prices(startd, endd, ticker)
    with_log = compute_log_returns(raw, freq=freq)
    with_both = compute_returns(with_log)

    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        for row in with_both:
            writer.writerow(row)

    print(f"Exported {len(with_both)-1} rows to {path}")
    return with_both

def days_30_360(start_date, end_date):
    """
    Calculate the number of days between two dates using the 30/360 US convention.
    Dates expected as 'YYYY-MM-DD' strings or datetime.date-like objects.

    Returns an integer day-count.
    """
    if isinstance(start_date, str):
        d1 = datetime.strptime(start_date, '%Y-%m-%d').date()
    else:
        d1 = start_date
    if isinstance(end_date, str):
        d2 = datetime.strptime(end_date, '%Y-%m-%d').date()
    else:
        d2 = end_date
    d1_day = min(d1.day, 30)
    # Adjust d2 day per 30/360 US rule
    if d2.day == 31 and d1_day == 30:
        d2_day = 30
    else:
        d2_day = min(d2.day, 30)
    return 360 * (d2.year - d1.year) + 30 * (d2.month - d1.month) + (d2_day - d1_day)


In [3]:
def prices_month(startd, endd, tic):
    info = []
    key = '&apikey=ZKMMTO1ATDBLXH2K'
    ticker = '&symbol=' + str(tic)
    endpoint = 'function=TIME_SERIES_MONTHLY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker + size + key

    r = requests.get(url)
    if r.status_code == 200:
        print('connection successful')
        data = r.json()
        r1 = data.get('Monthly Adjusted Time Series', {})
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info


def monthly_log_returns(startd, endd, tic):
    raw = prices_month(startd, endd, tic)
    return compute_log_returns(raw, freq='daily')


def sigma_matrix(tickers, startd, endd, window=60):
    """
    Downloads monthly data and estimates the monthly covariance matrix using
    the FIRST `window` returns (61 price observations → 60 returns).
    Returns (sigma_monthly, returns_df, prices_df).
    - returns_df: all downloaded returns (row index 0 = month 2 return, index 60 = month 62 return, ...)
    - prices_df:  prices aligned to same rows (index 0 = P_2, index 60 = P_62, ...)
    """
    series_ret = {}
    series_price = {}
    for i, tic in enumerate(tickers):
        if i > 0:
            time.sleep(3)  # free tier: 5 calls/min
        data = monthly_log_returns(startd, endd, tic)
        rows = [row for row in data if row[3] is not None]
        series_ret[tic]   = pd.Series({row[1]: row[3]          for row in rows})
        series_price[tic] = pd.Series({row[1]: float(row[2])   for row in rows})

    returns_df = pd.DataFrame(series_ret).dropna()
    prices_df  = pd.DataFrame(series_price).reindex(returns_df.index)
    df_train   = returns_df.iloc[:window]    # first 60 returns for estimation
    return df_train.cov(), returns_df, prices_df


def calculate_p0_p1_p2(prices_df, tickers):
    """
    Prices at months 62, 63, 64 for portfolio tickers.
    prices_df rows start at month 2 (first return row), so:
      iloc[60] = P_62, iloc[61] = P_63, iloc[62] = P_64.
    Appends cash price 1.0 so length matches tangent weights (which include a cash weight).
    """
    p0 = prices_df[tickers].iloc[60].values.reshape(-1, 1)  # P_62
    p1 = prices_df[tickers].iloc[61].values.reshape(-1, 1)  # P_63
    p2 = prices_df[tickers].iloc[62].values.reshape(-1, 1)  # P_64

    p0 = np.vstack([p0, [[1.0]]])
    p1 = np.vstack([p1, [[1.0]]])
    p2 = np.vstack([p2, [[1.0]]])
    return p0, p1, p2

In [4]:
tickers    = ["JPM", "AAPL", "MSFT", "GS", "BAC", "WMT", "MS"]
start_date = "2000-01-01"
end_date = "2020-12-31"
rf = 0.05           # annual risk-free rate GIVEN
rf_monthly = rf / 12
market = "SPY"

# Download 64-month batch; sigma estimated from first 60 returns (obs 1-61)
sigma, returns_df, prices_df = sigma_matrix(tickers + [market], start_date, end_date)

# --- Step 1: Monthly Sigma (first 60 returns) ---
print("Monthly Sigma Matrix:")
display(sigma.drop(market).drop(market, axis=1))

# --- Step 2: Beta (cov / var, from first 60 returns) ---
var_market = sigma.loc[market, market]
betas = sigma[market].drop(market) / var_market
print("\nBetas vs SPY:")
display(betas)

# --- Step 3: Month-62 realized market risk premium ---
# returns_df.iloc[60] = log(P_62 / P_61), i.e. the return from obs 61 → obs 62
R_market_62 = returns_df.iloc[60][market]
market_risk_premium = R_market_62 - rf_monthly
print(f"\nSPY return at month 62:         {R_market_62:.6f}")
print(f"Market risk premium (month 62): {market_risk_premium:.6f}")

# --- Step 4: CAPM expected monthly returns ---
mu = rf_monthly + betas * market_risk_premium
print("\nCAPM Expected Monthly Returns (for implementation at month 62):")
display(mu)

connection successful
connection successful
connection successful
connection successful
connection successful
connection successful
connection successful
connection successful
Monthly Sigma Matrix:


,JPM,AAPL,MSFT,GS,BAC,WMT,MS
JPM,0.012441,0.007704,0.005695,0.007405,0.005111,0.002062,0.009300
AAPL,0.007704,0.035497,0.009890,0.010177,0.002302,0.000086,0.009142
MSFT,0.005695,0.009890,0.014708,0.005057,0.001101,0.001851,0.005949
GS,0.007405,0.010177,0.005057,0.011090,0.002007,0.001127,0.010087
BAC,0.005111,0.002302,0.001101,0.002007,0.005099,0.000744,0.003054
WMT,0.002062,0.000086,0.001851,0.001127,0.000744,0.004501,0.001926
MS,0.009300,0.009142,0.005949,0.010087,0.003054,0.001926,0.012388



Betas vs SPY:


JPM     1.822798
AAPL    1.980926
MSFT    1.521774
GS      1.597605
BAC     0.680215
WMT     0.501625
MS      1.960809
Name: SPY, dtype: float64


SPY return at month 62:         0.020689
Market risk premium (month 62): 0.016522

CAPM Expected Monthly Returns (for implementation at month 62):


JPM     0.034283
AAPL    0.036895
MSFT    0.029309
GS      0.030562
BAC     0.015405
WMT     0.012455
MS      0.036563
Name: SPY, dtype: float64

In [5]:
# Tangent portfolio — CAPM mu, monthly sigma
# For implementation at the start of month 62

sigma_assets = sigma.drop(market).drop(market, axis=1)
excess = mu.values - rf_monthly
sigma_inv = np.linalg.inv(sigma_assets.values)

z = sigma_inv @ excess
T = z / z.sum()

tangent = pd.Series(T, index=mu.index)
print("Tangent Portfolio Weights (for implementation at month 62):")
display(tangent)

Tangent Portfolio Weights (for implementation at month 62):


JPM     0.168482
AAPL    0.051882
MSFT    0.158033
GS     -0.012442
BAC     0.085581
WMT     0.149524
MS      0.398940
dtype: float64

In [6]:
def initial_values(tangent_w, p0, k0, I):
    w0 = tangent_w.values.reshape(-1, 1)
    n = len(w0.ravel())

    sign = np.sign(w0).ravel()
    D = np.diag(sign / k0.ravel())
    ones = np.ones((1, n))
    zero = np.array([[0]])

    A = np.block([[D, -1*w0], [ones, zero]])
    C = np.zeros((n + 1, 1))
    C[-1, 0] = I

    m = np.linalg.inv(A) @ C
    m0 = m[0:n].round(3)
    asset_sum = m[n:].round(3)

    if np.sum(m0) > I + (I/100) or np.sum(m0) < I - (I/100):
        print("Margin Allocations != I")

    e0 = m0
    a0 = (e0/k0) * np.sign(w0)
    l0 = np.where(w0>0, a0-e0, a0*-1)
    s0 = a0 / p0
    er0 = k0
    c0 = np.array([0.0, 0.0])

    f0 = np.zeros((n, 1))

    print("----------  Initial Values  ----------")
    print("# Shares",    s0.ravel().round(3))
    print("$ Assets",    a0.ravel().round(3))
    print("$ Equity",    e0.ravel().round(3))
    print("$ Liability", l0.ravel().round(3))
    print("E Ratio",     er0.ravel().round(3))
    print("$ Margin",    m0.ravel().round(3))
    print(f"Unlocked C {c0[0]} - Locked C {c0[1]}")

    return s0, a0, e0, l0, er0, m0, c0, f0


In [7]:
def price_change(p0, p1, s, a, e, l, er, m, c, f, long_fee, short_fee):
    a1 = p1 * s
    l1 = np.where(a1>0, l, l + (p0-p1)*s)
    e1 = np.where(a1>0, e + (a1 - a), e - (p0-p1)*s)
    er1 = e1 / a1 * np.sign(a1)

    todays_long_fee  = np.where(a>0, l, 0) * long_fee
    todays_short_fee = np.where(a<0, l, 0) * short_fee
    todays_total_fee = todays_long_fee + todays_short_fee
    e1 = e1 - todays_total_fee

    f1 = f + todays_total_fee

    print("----------  After Price Change  ----------")
    print("# Shares",    s.ravel().round(3))
    print("$ Assets",    a1.ravel().round(3))
    print("$ Equity",    e1.ravel().round(3))
    print("$ Liability", l1.ravel().round(3))
    print("E Ratio",     er1.ravel().round(3))
    print("$ Margin",    m.ravel().round(3))
    print("$ Fees",      f1.ravel().round(3))
    print(f"Unlocked C {c[0]} - Locked C {c[1]}")

    return s, a1, e1, l1, er1, m, c, f1


In [8]:
def margin_call(s, a, e, l, er, m, c, f):
    ulcash = c[0]
    lcash  = c[1]

    mc   = np.where(er < 0.25, "YES", "NO")
    en   = 0.25 * a * np.sign(a)                    # minimum equity needed per position
    eadd = np.where(er < 0.25, en - e, 0)           # equity to add per position

    print("Equity Right Now");       print(e.ravel().round(3))
    print("Total Equity Required");  print(en.ravel().round(3))
    print("Equity Need to Add");     print(eadd.ravel().round(3))

    if np.sum(eadd) > 0:
        print("Need Margin Call")
        lcash  += np.sum(eadd)
        e       = e + eadd
        ulcash -= np.sum(eadd)
        m       = m + eadd

        if ulcash < 0:
            cadd = -1 * ulcash
            print(f"We need to add ${float(np.asarray(cadd).squeeze()):.2f} of cash")
            ulcash = 0
        else:
            print("Did not need to add more cash")

        c = np.array([float(np.asarray(ulcash).squeeze()),
                      float(np.asarray(lcash).squeeze())], dtype=float)
    else:
        print("No margin call needed")

    er = e / a * np.sign(a)

    print("\n----------  After Margin Call  ----------")
    print("# Shares",    s.ravel().round(3))
    print("$ Assets",    a.ravel().round(3))
    print("$ Equity",    e.ravel().round(3))
    print("$ Liability", l.ravel().round(3))
    print("E Ratio",     er.ravel().round(3))
    print("$ Margin",    m.ravel().round(3))
    print(f"Unlocked C {round(c[0], 2)} - Locked C {round(c[1], 2)}")

    return s, a, e, l, er, m, c, f


In [9]:
def rebalance2(p, s, a, e, l, er, m, c, f, tangent_w):
    w0 = tangent_w.values.reshape(-1, 1)

    w1   = a / a.sum()
    dist = np.linalg.norm(w1 - w0) / np.linalg.norm(w0)

    if dist > 0.05:
        print(f"Distance is {dist} YES rebalance (rebalance2).")

        V      = np.sum(a)
        a_star = w0 * V
        d      = a_star - a

        a1 = a + d  # lam=1 always — no cash position to constrain rebalance

        l1 = np.where(a1>0, l + ((a1 - a) * (1 - er)), np.abs(a1))
        e1 = np.where(a1>0, e + ((a1 - a) * (er)),      a1 * er)
        e1 = np.abs(e1)
        s1 = a1 / p

        delta_s   = s1 - s
        rebal_val = delta_s * p
        m1 = np.abs(np.where(w1>0, m + rebal_val * er, m - rebal_val * er))

        c1 = c  # no cash position, c unchanged
    else:
        a1 = a
        e1 = e
        s1 = s
        l1 = l
        m1 = m
        c1 = c
        print(f"Distance is {dist} NO rebalance.")

    print("\n----------  After Rebalance  ----------")
    print("# Shares",    s1.ravel().round(3))
    print("$ Assets",    a1.ravel().round(3))
    print("$ Equity",    e1.ravel().round(3))
    print("$ Liability", l1.ravel().round(3))
    print("E Ratio",     er.ravel().round(3))
    print("$ Margin",    m1.ravel().round(3))
    print(f"Unlocked C {round(c1[0], 2)} - Locked C {round(c1[1], 2)}")

    return s1, a1, e1, l1, er, m1, c1, f


In [10]:
p0, p1, p2 = calculate_p0_p1_p2(prices_df, tickers)

I  = 10000
k0 = np.full((len(tangent), 1), 0.5)

s0, a0, e0, l0, er0, m0, c0, f0 = initial_values(tangent, p0[:-1], k0, I)


----------  Initial Values  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3287.83  1012.456 3083.92  -242.792 1670.054 2917.872 7785.078]
$ Equity [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
$ Liability [1643.915  506.228 1541.96   242.792  835.027 1458.936 3892.539]
E Ratio [0.5 0.5 0.5 0.5 0.5 0.5 0.5]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
Unlocked C 0.0 - Locked C 0.0


In [11]:
long_fee  = 0.0001 
short_fee = 0.0001

s, a1, e1, l1, er1, m, c, f1 = price_change(p0[:-1], p1[:-1], s0, a0, e0, l0, er0, m0, c0, f0, long_fee, short_fee)


----------  After Price Change  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3112.427  940.444 2962.578 -245.447 1594.018 2841.341 7892.605]
$ Equity [1468.348  434.166 1420.464  118.716  758.907 1382.259 3999.677]
$ Liability [1643.915  506.228 1541.96   245.447  835.027 1458.936 3892.539]
E Ratio [0.472 0.462 0.48  0.484 0.476 0.487 0.507]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
$ Fees [0.164 0.051 0.154 0.024 0.084 0.146 0.389]
Unlocked C 0.0 - Locked C 0.0


In [12]:
s, a1, e1, l1, er1, m0, c0, f1 = margin_call(s, a1, e1, l1, er1, m0, c0, f1)


Equity Right Now
[1468.348  434.166 1420.464  118.716  758.907 1382.259 3999.677]
Total Equity Required
[ 778.107  235.111  740.645   61.362  398.504  710.335 1973.151]
Equity Need to Add
[0. 0. 0. 0. 0. 0. 0.]
No margin call needed

----------  After Margin Call  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3112.427  940.444 2962.578 -245.447 1594.018 2841.341 7892.605]
$ Equity [1468.348  434.166 1420.464  118.716  758.907 1382.259 3999.677]
$ Liability [1643.915  506.228 1541.96   245.447  835.027 1458.936 3892.539]
E Ratio [0.472 0.462 0.479 0.484 0.476 0.486 0.507]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
Unlocked C 0.0 - Locked C 0.0


In [13]:
s, a1, e1, l1, er1, m0, c0, f1 = rebalance2(p1[:-1], s, a1, e1, l1, er1, m0, c0, f1, tangent)


Distance is 0.032342444103795366 NO rebalance.

----------  After Rebalance  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3112.427  940.444 2962.578 -245.447 1594.018 2841.341 7892.605]
$ Equity [1468.348  434.166 1420.464  118.716  758.907 1382.259 3999.677]
$ Liability [1643.915  506.228 1541.96   245.447  835.027 1458.936 3892.539]
E Ratio [0.472 0.462 0.479 0.484 0.476 0.486 0.507]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
Unlocked C 0.0 - Locked C 0.0


In [14]:
s, a, e, l, er, m, c, f = price_change(p1[:-1], p2[:-1], s0, a0, e0, l0, er0, m0, c0, f0, long_fee, short_fee)


----------  After Price Change  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3224.624  813.821 3101.092 -238.878 1627.996 2672.953 7291.177]
$ Equity [1580.545  307.543 1558.978  127.941  792.886 1213.871 3398.249]
$ Liability [1643.915  506.228 1541.96   236.223  835.027 1458.936 3892.539]
E Ratio [0.49  0.378 0.503 0.536 0.487 0.454 0.466]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
$ Fees [0.164 0.051 0.154 0.024 0.084 0.146 0.389]
Unlocked C 0.0 - Locked C 0.0


In [15]:
s, a, e, l, er, m, c, f = price_change(p1[:-1], p2[:-1], s0, a0, e0, l0, er0, m0, c0, f, long_fee, short_fee)


----------  After Price Change  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3224.624  813.821 3101.092 -238.878 1627.996 2672.953 7291.177]
$ Equity [1580.545  307.543 1558.978  127.941  792.886 1213.871 3398.249]
$ Liability [1643.915  506.228 1541.96   236.223  835.027 1458.936 3892.539]
E Ratio [0.49  0.378 0.503 0.536 0.487 0.454 0.466]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
$ Fees [0.329 0.101 0.308 0.049 0.167 0.292 0.779]
Unlocked C 0.0 - Locked C 0.0


In [16]:
s, a, e, l, er, m, c, f = rebalance2(p2[:-1], s, a, e, l, er, m, c, f, tangent)


Distance is 0.031482908094677985 NO rebalance.

----------  After Rebalance  ----------
# Shares [157.425 753.259 177.764  -3.101  60.567 258.027 258.168]
$ Assets [3224.624  813.821 3101.092 -238.878 1627.996 2672.953 7291.177]
$ Equity [1580.545  307.543 1558.978  127.941  792.886 1213.871 3398.249]
$ Liability [1643.915  506.228 1541.96   236.223  835.027 1458.936 3892.539]
E Ratio [0.49  0.378 0.503 0.536 0.487 0.454 0.466]
$ Margin [1643.915  506.228 1541.96   121.396  835.027 1458.936 3892.539]
Unlocked C 0.0 - Locked C 0.0
